# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Optional external author name disambiguation
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation


---
## Setup

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np

# Track run timing
_pipeline_start = time.time()

from ads_bib.config import init_paths, load_env
from ads_bib.pipeline import (
    PipelineConfig,
    PipelineContext,
    STAGE_ORDER,
    run_author_disambiguation_stage,
    run_citations_stage,
    run_curate_stage,
    run_embeddings_stage,
    run_export_stage,
    run_reduction_stage,
    run_search_stage,
    run_tokenize_stage,
    run_topic_dataframe_stage,
    run_topic_fit_stage,
    run_translate_stage,
    run_visualize_stage,
    validate_stage_name,
)
from ads_bib.run_manager import RunManager
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses notebook/project root
load_env()
tracker = CostTracker()
run = RunManager(run_name="ADS_Curation_Run")
ctx = None
_last_pipeline_config_data = None


def build_pipeline_config() -> PipelineConfig:
    return PipelineConfig.from_dict(
        {
            "run": {
                "run_name": globals().get("RUN_NAME", "ADS_Curation_Run"),
                "start_stage": globals().get("START_STAGE", "search"),
                "stop_stage": globals().get("STOP_STAGE"),
                "random_seed": globals().get("RANDOM_SEED", 42),
                "openrouter_cost_mode": globals().get("OPENROUTER_COST_MODE", "hybrid"),
                "project_root": str(paths["project_root"]),
            },
            "search": {
                "query": globals().get("QUERY", ""),
                "ads_token": globals().get("ADS_TOKEN"),
                "refresh_search": globals().get("REFRESH_SEARCH", True),
                "refresh_export": globals().get("REFRESH_EXPORT", True),
            },
            "translate": {
                "enabled": globals().get("TRANSLATION_ENABLED", True),
                "provider": globals().get("TRANSLATION_PROVIDER", "openrouter"),
                "model": globals().get("TRANSLATION_MODEL", "google/gemini-3-flash-preview"),
                "api_key": globals().get("TRANSLATION_API_KEY"),
                "max_workers": globals().get("TRANSLATION_MAX_WORKERS", 10),
                "max_tokens": globals().get("TRANSLATION_MAX_TOKENS", 2048),
                "fasttext_model": globals().get("FASTTEXT_MODEL"),
            },
            "tokenize": {
                "enabled": globals().get("TOKENIZATION_ENABLED", True),
                "spacy_model": globals().get("TOKENIZATION_SPACY_MODEL", "en_core_web_lg"),
                "batch_size": globals().get("TOKENIZATION_BATCH_SIZE", 512),
                "n_process": globals().get("TOKENIZATION_N_PROCESS", 1),
                "disable": globals().get("TOKENIZATION_DISABLE", ("ner", "parser", "textcat")),
                "fallback_model": globals().get("TOKENIZATION_FALLBACK_MODEL", "en_core_web_lg"),
                "auto_download": globals().get("TOKENIZATION_AUTO_DOWNLOAD", True),
            },
            "author_disambiguation": {
                "enabled": globals().get("ENABLE_AUTHOR_DISAMBIGUATION", False),
                "model_bundle": globals().get("AND_MODEL_BUNDLE"),
                "dataset_id": globals().get("AND_DATASET_ID"),
                "force_refresh": globals().get("AND_FORCE_REFRESH", False),
                "infer_stage": globals().get("AND_INFER_STAGE", "full"),
            },
            "topic_model": {
                "sample_size": globals().get("SAMPLE_SIZE"),
                "embedding_provider": globals().get("EMBEDDING_PROVIDER", "openrouter"),
                "embedding_model": globals().get("EMBEDDING_MODEL", "google/gemini-embedding-001"),
                "embedding_api_key": globals().get("EMBEDDING_API_KEY"),
                "embedding_batch_size": globals().get("EMBEDDING_BATCH_SIZE", 64),
                "embedding_max_workers": globals().get("EMBEDDING_MAX_WORKERS", 20),
                "reduction_method": globals().get("DIM_REDUCTION_METHOD", "pacmap"),
                "params_5d": globals().get("PARAMS_5D", {}),
                "params_2d": globals().get("PARAMS_2D", {}),
                "backend": globals().get("TOPIC_BACKEND", "bertopic"),
                "clustering_method": globals().get("CLUSTERING_METHOD", "fast_hdbscan"),
                "cluster_params": globals().get("CLUSTER_PARAMS", {}),
                "toponymy_cluster_params": globals().get("TOPONYMY_CLUSTER_PARAMS", {}),
                "toponymy_evoc_cluster_params": globals().get("TOPONYMY_EVOC_CLUSTER_PARAMS", {}),
                "toponymy_layer_index": globals().get("TOPONYMY_LAYER_INDEX", 1),
                "llm_prompt": globals().get("LLM_PROMPT"),
                "llm_provider": globals().get("LLM_PROVIDER", "openrouter"),
                "llm_model": globals().get("LLM_MODEL", "google/gemini-3-flash-preview"),
                "llm_api_key": globals().get("LLM_API_KEY"),
                "bertopic_label_max_tokens": globals().get("BERTOPIC_LABEL_MAX_TOKENS", 128),
                "toponymy_local_label_max_tokens": globals().get("TOPONYMY_LOCAL_LABEL_MAX_TOKENS", 256),
                "pipeline_models": globals().get("PIPELINE_MODELS", ["POS", "KeyBERT", "MMR"]),
                "parallel_models": globals().get("PARALLEL_MODELS", ["MMR", "POS", "KeyBERT"]),
                "toponymy_embedding_model": globals().get("TOPONYMY_EMBEDDING_MODEL"),
                "toponymy_max_workers": globals().get("TOPONYMY_MAX_WORKERS", 10),
                "min_df": globals().get("MIN_DF"),
                "outlier_threshold": globals().get("OUTLIER_THRESHOLD", 0.5),
            },
            "visualization": {
                "enabled": globals().get("VISUALIZATION_ENABLED", True),
                "title": globals().get("VISUALIZATION_TITLE", "ADS Bibliometric Map"),
                "subtitle_template": globals().get(
                    "VISUALIZATION_SUBTITLE_TEMPLATE",
                    "Topics labeled with {provider}/{model}",
                ),
                "dark_mode": globals().get("VISUALIZATION_DARK_MODE", True),
            },
            "curation": {
                "clusters_to_remove": globals().get("CLUSTERS_TO_REMOVE", []),
            },
            "citations": {
                "metrics": globals().get(
                    "CITE_METRICS",
                    ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"],
                ),
                "min_counts": globals().get("MIN_COUNTS", {}),
                "authors_filter": globals().get("AUTHORS_FILTER"),
                "output_format": globals().get("OUTPUT_FORMAT", "gexf"),
            },
        }
    )


def _nested_get(data: dict, path: tuple[str, ...]):
    current = data
    for key in path:
        if not isinstance(current, dict):
            return None
        current = current.get(key)
    return current


def _earliest_invalidation_stage(previous: dict | None, current: dict) -> str | None:
    if previous is None:
        return None

    stage_checks = [
        ("search", [("search",)]),
        ("translate", [("translate",), ("run", "openrouter_cost_mode")]),
        ("tokenize", [("tokenize",)]),
        ("author_disambiguation", [("author_disambiguation",)]),
        (
            "embeddings",
            [
                ("run", "random_seed"),
                ("topic_model", "sample_size"),
                ("topic_model", "embedding_provider"),
                ("topic_model", "embedding_model"),
                ("topic_model", "embedding_api_key"),
                ("topic_model", "embedding_batch_size"),
                ("topic_model", "embedding_max_workers"),
            ],
        ),
        (
            "reduction",
            [
                ("topic_model", "reduction_method"),
                ("topic_model", "params_5d"),
                ("topic_model", "params_2d"),
            ],
        ),
        (
            "topic_fit",
            [
                ("topic_model", "backend"),
                ("topic_model", "clustering_method"),
                ("topic_model", "cluster_params"),
                ("topic_model", "toponymy_cluster_params"),
                ("topic_model", "toponymy_evoc_cluster_params"),
                ("topic_model", "toponymy_layer_index"),
                ("topic_model", "llm_prompt"),
                ("topic_model", "llm_provider"),
                ("topic_model", "llm_model"),
                ("topic_model", "llm_api_key"),
                ("topic_model", "bertopic_label_max_tokens"),
                ("topic_model", "toponymy_local_label_max_tokens"),
                ("topic_model", "pipeline_models"),
                ("topic_model", "parallel_models"),
                ("topic_model", "toponymy_embedding_model"),
                ("topic_model", "toponymy_max_workers"),
                ("topic_model", "min_df"),
                ("topic_model", "outlier_threshold"),
            ],
        ),
        ("visualize", [("visualization",)]),
        ("curate", [("curation",)]),
        ("citations", [("citations",)]),
    ]

    for stage, paths_to_check in stage_checks:
        if any(_nested_get(previous, path) != _nested_get(current, path) for path in paths_to_check):
            return stage
    return None


def invalidate_context_from(context: PipelineContext, stage: str) -> None:
    stage_name = validate_stage_name(stage)
    stage_index = STAGE_ORDER.index(stage_name)

    if stage_index <= STAGE_ORDER.index("search"):
        context.bibcodes = None
        context.references = None
        context.esources = None
        context.fulltext_urls = None
        context.publications = None
        context.refs = None
    elif stage_index <= STAGE_ORDER.index("author_disambiguation"):
        context.publications = None
        context.refs = None

    if stage_index <= STAGE_ORDER.index("embeddings"):
        context.topic_input_df = None
        context.documents = None
        context.embeddings = None
        context.reduced_5d = None
        context.reduced_2d = None
        context.topic_model = None
        context.topics = None
        context.topic_info = None
        context.topic_df = None
        context.curated_df = None
        context.citation_results = None
        return

    if stage_index <= STAGE_ORDER.index("reduction"):
        context.reduced_5d = None
        context.reduced_2d = None
        context.topic_model = None
        context.topics = None
        context.topic_info = None
        context.topic_df = None
        context.curated_df = None
        context.citation_results = None
        return

    if stage_index <= STAGE_ORDER.index("topic_fit"):
        context.topic_model = None
        context.topics = None
        context.topic_info = None
        context.topic_df = None
        context.curated_df = None
        context.citation_results = None
        return

    if stage_index <= STAGE_ORDER.index("topic_dataframe"):
        context.topic_df = None
        context.curated_df = None
        context.citation_results = None
        return

    if stage_index <= STAGE_ORDER.index("curate"):
        context.curated_df = None
        context.citation_results = None
        return

    if stage_index <= STAGE_ORDER.index("citations"):
        context.citation_results = None


def current_context(reset: bool = False) -> PipelineContext:
    global ctx, _last_pipeline_config_data
    config = build_pipeline_config()
    config_data = config.to_dict()
    random.seed(config.run.random_seed)
    np.random.seed(config.run.random_seed)
    if reset or ctx is None:
        ctx = PipelineContext.create(
            config,
            project_root=paths["project_root"],
            paths=paths,
            run=run,
            tracker=tracker,
            start_time=_pipeline_start,
            load_environment=False,
        )
    else:
        invalidation_stage = _earliest_invalidation_stage(_last_pipeline_config_data, config_data)
        if invalidation_stage is not None:
            invalidate_context_from(ctx, invalidation_stage)
            logger.info(
                "Config changed; invalidated in-memory state from stage '%s'.",
                invalidation_stage,
            )
        ctx.config = config
    _last_pipeline_config_data = config_data
    run.save_config(config)
    return ctx


def stage_enabled(stage: str) -> bool:
    stage_name = validate_stage_name(stage)
    start = validate_stage_name(globals().get("START_STAGE", "search"))
    stop = globals().get("STOP_STAGE")
    stage_index = STAGE_ORDER.index(stage_name)
    if stage_index < STAGE_ORDER.index(start):
        return False
    if stop is not None:
        stop_name = validate_stage_name(stop)
        if stage_index > STAGE_ORDER.index(stop_name):
            return False
    return True


def sync_notebook_state(context: PipelineContext) -> None:
    globals()["publications"] = context.publications
    globals()["refs"] = context.refs
    globals()["df"] = context.curated_df if context.curated_df is not None else context.topic_df
    globals()["documents"] = context.documents
    globals()["embeddings"] = context.embeddings
    globals()["reduced_5d"] = context.reduced_5d
    globals()["reduced_2d"] = context.reduced_2d
    globals()["topic_model"] = context.topic_model
    globals()["topics"] = context.topics
    globals()["topic_info"] = context.topic_info
    globals()["results"] = context.citation_results


def execute_stage(stage: str, runner) -> PipelineContext | None:
    if not stage_enabled(stage):
        logger.info("Skipping Stage %s (START_STAGE=%s, STOP_STAGE=%s)", stage, START_STAGE, STOP_STAGE)
        return None
    context = current_context()
    runner(context)
    sync_notebook_state(context)
    return context


In [ ]:
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")

if root_logger.handlers:
    for handler in root_logger.handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
else:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

logger = logging.getLogger("pipeline")

---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
# Named resume points: search, export, translate, tokenize,
# author_disambiguation, embeddings, reduction, topic_fit,
# topic_dataframe, visualize, curate, citations
RUN_NAME = "ADS_Curation_Run"
START_STAGE = "search"
STOP_STAGE = None
REFRESH_SEARCH = True
REFRESH_EXPORT = True

# Optional external AND step
ENABLE_AUTHOR_DISAMBIGUATION = False
AND_MODEL_BUNDLE = None
AND_DATASET_ID = None
AND_FORCE_REFRESH = False
AND_INFER_STAGE = "full"

# Deterministic seed for notebook-side randomness
RANDOM_SEED = 42

# Shared OpenRouter pricing mode
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'
# ──────────────────────────────────────────────────────

ctx = current_context(reset=True)


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits for your research question.
These values define the corpus scope for all downstream steps.


In [ ]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
Set_B = f"citations({Set_A}) AND year:1915-2000"
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gravity_relativity_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

# Example quick focus query
# QUERY = 'author:"Treder, H*"'
# Alternative broader query:
QUERY = f"({Set_D}) OR ({Set_T}) OR ({Set_E})"

In [ ]:
# # ...existing code...
# # --- SEARCH CONFIGURATION ---
# # Modify the query string for your research question.
# # See: https://ui.adsabs.harvard.edu/help/search/search-syntax

# ADS_TOKEN = os.getenv("ADS_API_KEY")

# # 1. Historischer Seed: Einsteins Publikationen (1911-1955)
# Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"

# # 2. Direkte Rezeption: 1-Hop Zitationen ab 1915
# Set_B = f"citations({Set_A}) AND year:1915-2000"

# # 3. Strukturelle Anker-Begriffe (Mehrsprachig: EN, DE, FR, IT)
# generic_anchors = [
#     'relativ*', 'einstein*', 
#     'spacetime', '"space-time"', 'raumzeit', '"espace-temps"', 'spaziotempo', '"spazio-tempo"',
#     'tensor*', 'tenseur*', 
#     'metric*', 'metrik*', 'metriqu*', 
#     'curvature', 'krümmung', 'kruemmung', 'courbure', 'curvatura',
#     'geodesic*', 'geodät*', 'geodaet*', 'geodesiqu*', 'geodetic*'
# ]

# # 4. Indirekte Rezeption: 2-Hop Zitationen ab 1915 (Gefiltert durch Anker)
# Set_C = f"citations(citations({Set_A})) AND year:1915-2000 AND abs:({' OR '.join(generic_anchors)})"

# # 5. Treder Library
# Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

# # 6. Text-Suche: Core-Begriffe und verankerte breite Begriffe
# grg_core_terms = [
#     '(general AND relativi*)', '(allgemein* AND relativi*)',
#     '"relativité générale"', '"relatività generale"',
#     '"Gravité quantique"', '"Gravità quantistica"', 'Quantengravitation', '"quantum gravity"',
#     '(einheitlich* AND feld*)', '"champ unifié"', '(unified AND field*)'
# ]

# broad_terms = ['gravit*', 'cosmo*', 'Kosmo*']
# anchored_broad = f"({' OR '.join(broad_terms)}) AND ({' OR '.join(generic_anchors)})"

# Set_E = f"abs:({' OR '.join(grg_core_terms)} OR ({anchored_broad})) AND year:1911-2000"

# # Finale Query
# QUERY = f"({Set_A}) OR ({Set_B}) OR ({Set_C}) OR ({Set_T}) OR ({Set_E})"

### 1.2 Execute Search

Run the ADS search and persist bibcodes/results to this run folder.
This creates a reproducible input snapshot for later phases.


In [ ]:
ctx = execute_stage("search", run_search_stage)
if ctx is not None:
    bibcodes = ctx.bibcodes
    references = ctx.references
    esources = ctx.esources
    fulltext_urls = ctx.fulltext_urls


### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.
This normalizes schemas before translation/tokenization/topic modeling.


In [ ]:
ctx = execute_stage("export", run_export_stage)
if ctx is not None:
    publications = ctx.publications
    refs = ctx.refs


In [ ]:
if stage_enabled("export") and globals().get("publications") is not None:
    preview_cols = [
        c for c in ("Bibcode", "Year", "Author", "Title", "References")
        if c in publications.columns
    ]
    logger.info(
        "Phase 1 preview: publications=%s rows, columns=%s",
        f"{len(publications):,}",
        len(publications.columns),
    )
    if preview_cols:
        display(publications[preview_cols].head(10))
    else:
        display(publications.head(10))

---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider/model and translation target language.
Keep settings explicit so costs and outputs are reproducible.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
# Providers:
#   'openrouter' — API, any LLM (requires OPENROUTER_API_KEY)
#   'gguf'       — Local GPU via llama-cpp-python (TranslateGemma etc.)
#   'nllb'       — Local CPU via CTranslate2+NLLB (fast, 200+ languages)
TRANSLATION_PROVIDER = "openrouter"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"
# GGUF alternative (recommended for GPU):
# TRANSLATION_PROVIDER = "openrouter"
# TRANSLATION_MODEL = "google/gemini-3-flash-preview"
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 10
TRANSLATION_MAX_TOKENS = 2048

# Path to fasttext language ID model:
# https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"
# ──────────────────────────────────────────────────────

### 2.2 Language Detection

Detect language tags for configured text columns.
Only non-target-language rows are translated in the next step.


In [ ]:
ctx = execute_stage("translate", run_translate_stage)
if ctx is not None:
    publications = ctx.publications
    refs = ctx.refs


### 2.3 Translate Non-English Entries

Translate non-English rows and track token/cost metadata.
This harmonizes text fields for downstream NLP and topic modeling.


In [ ]:
ctx = execute_stage("translate", run_translate_stage)
if ctx is not None and publications is not None:
    preview_cols = [c for c in ("Title", "Title_en", "Abstract", "Abstract_en") if c in publications.columns]
    if preview_cols:
        display(publications[preview_cols].head(5))


### 2.4 Save Translation Checkpoint

Persist translated data to global cache and run folder for Phase 3 restarts.

In [ ]:
if stage_enabled("translate"):
    ctx = current_context()
    logger.info("Translated snapshot managed by shared runner under %s", paths["cache"])


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy.
References are intentionally skipped in this phase for runtime stability.

In [ ]:
import os

# ── CONFIGURE ─────────────────────────────────────────
TOKENIZATION_SPACY_MODEL = "en_core_web_lg"
TOKENIZATION_BATCH_SIZE = 512
TOKENIZATION_N_PROCESS = min(max((os.cpu_count() or 1) - 1, 1), 8)
TOKENIZATION_DISABLE = ("ner", "parser", "textcat")
# ──────────────────────────────────────────────────────

ctx = execute_stage("tokenize", run_tokenize_stage)
if ctx is not None:
    publications = ctx.publications
    refs = ctx.refs


In [ ]:
if stage_enabled("tokenize") and globals().get("refs") is not None:
    logger.info("References dataset available at %s", run.paths["data"] / "references.parquet")


### Checkpoint: Save Phase 3 Results

Persist tokenized publications and translated references for Phase 4 restarts.
This avoids rerunning API-heavy earlier phases.


In [ ]:
if stage_enabled("tokenize"):
    ctx = current_context()
    logger.info("Tokenized snapshot managed by shared runner under %s", paths["cache"])


---
# Phase 4: Author Name Disambiguation

Optionally run an external AND package on the curated source datasets.
If disabled, the pipeline stores a Phase-4 checkpoint unchanged and continues.


In [ ]:
execute_stage("author_disambiguation", run_author_disambiguation_stage)


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling.
These settings strongly affect topic quality and runtime/cost.

| Provider | Model Examples | Cost | Notes |
|----------|----------------|------|-------|
| `local` | `google/embeddinggemma-300m`, `BAAI/bge-large-en-v1.5` | None | CPU/GPU, no API needed |
| `huggingface_api` | `huggingface/BAAI/bge-large-en-v1.5` | Per-token | HF Inference API via LiteLLM |
| `openrouter` | `openai/text-embedding-3-large`, `google/gemini-embedding-001` | Per-token | Central billing, thread-pool concurrency |

**Caching**: Embeddings are cached to `data/cache/embeddings/` with SHA-256 fingerprint validation.
Changing the dataset or model triggers automatic recomputation.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
# Providers: 'openrouter' (API), 'huggingface_api' (remote via LiteLLM), 'local' (SentenceTransformers)
EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")
EMBEDDING_BATCH_SIZE = 64
EMBEDDING_MAX_WORKERS = 20

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None
# ──────────────────────────────────────────────────────

### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.
Caching keeps repeated runs fast and reproducible.


In [ ]:
logger.info(
    "Topic stages resume via the shared pipeline runner and stage snapshots."
)


In [ ]:
execute_stage("embeddings", run_embeddings_stage)


### 5.3 Dimensionality Reduction Configuration

Two projections are computed: **5D** (HDBSCAN clustering input) and **2D** (visualization & KDE).

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| `pacmap` | Fast, good local/global balance | No `densmap` (density-preserving mode) |
| `umap` | Supports `densmap=True` for KDE, better hierarchical structure | Slower, sensitive to `n_neighbors` |

**Key parameters:**

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `n_neighbors` | 80 (PaCMAP) / 80 (UMAP) | Higher = more global structure; lower = more local detail |
| `min_dist` | 0.05 (UMAP only) | Lower = tighter clusters in 2D. Library default 0.1 is too loose for bibliometric data |
| `metric` | `angular`/`cosine` | PaCMAP uses `angular` (auto-converted from `cosine`) |
| `densmap` | `False` (UMAP only) | Set `True` in `PARAMS_2D` if you plan KDE density analysis downstream |

> **Tip**: If you plan KDE analysis on the 2D coordinates, use `method="umap"` with
> `PARAMS_2D = dict(..., densmap=True)`. Without densmap, UMAP inflates dense regions
> in the 2D projection, distorting density estimates.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
DIM_REDUCTION_METHOD = "pacmap"  # PaCMAP als lokaler/globaler Kompromiss für großes GRG-Korpus

PARAMS_5D = dict(n_neighbors=80, metric="angular", random_state=RANDOM_SEED)
PARAMS_2D = dict(n_neighbors=80, metric="angular", random_state=RANDOM_SEED)
# ──────────────────────────────────────────────────────

In [ ]:
execute_stage("reduction", run_reduction_stage)


### 5.4 Clustering Configuration

HDBSCAN discovers topic clusters based on density in the 5D space.

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `MIN_CLUSTER_SIZE` | `max(15, n_docs * 0.001)` | Controls granularity: ~0.1% of docs. Lower = more (smaller) topics |
| `min_samples` | 2–3 | Lower = fewer outliers but noisier clusters. Higher = stricter density |
| `cluster_selection_method` | `"eom"` | Excess of Mass: selects most persistent clusters |
| `cluster_selection_epsilon` | 0.02–0.05 | Absorbs border points; higher = larger clusters, fewer outliers |

**Backend choice:**
- `fast_hdbscan`: Fastest, but no `prediction_data` or `gen_min_span_tree` (no `approximate_predict()`).
- `hdbscan`: Supports `prediction_data=True` for predicting new documents and `gen_min_span_tree=True` for hierarchy analysis.

`BASE_MIN_CLUSTER_SIZE` is only used by Toponymy for micro-cluster granularity.


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
CLUSTERING_METHOD = "fast_hdbscan"  # Schnellste Variante für 183k

n_docs = len(df)
MIN_CLUSTER_SIZE = max(15, int(n_docs * 0.001)) # ca. 184 bei 183k Dokumenten
BASE_MIN_CLUSTER_SIZE = max(55, int(n_docs * 0.0007))
# ──────────────────────────────────────────────────────

logger.info("Dataset: %s documents", f"{n_docs:,}")
logger.info("  MIN_CLUSTER_SIZE=%s", MIN_CLUSTER_SIZE)
logger.info("  BASE_MIN_CLUSTER_SIZE=%s", BASE_MIN_CLUSTER_SIZE)

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=3, # Strikter für 183k: stabilere Kernpunkte, weniger Rauschen in Übergangszonen
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.05, # Absorbiert Randpunkte in bestehende Cluster
)

TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=3,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

# EVoC uses a backend-specific constructor in some versions.
# Keep this dict conservative; unsupported keys are filtered in fit_toponymy.
TOPONYMY_EVOC_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=3,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
    noise_level=0.35,
    n_neighbors=15,
    n_epochs=35,
)

TOPONYMY_LAYER_INDEX = 1  # Broader Toponymy layer to reduce -1 outliers on large corpora.

### 5.5 Backend & LLM Configuration

**Backend matrix:**

| Backend | Clustering Input | LLM Providers | Notes |
|---------|-----------------|---------------|-------|
| `bertopic` | 5D reduced vectors | `local`, `gguf`, `huggingface_api`, `openrouter` | Standard BERTopic + outlier reduction |
| `toponymy` | 5D reduced vectors | `local`, `gguf`, `openrouter` | Hierarchical layers, richer labeling |
| `toponymy_evoc` | Raw embeddings | `local`, `gguf`, `openrouter` | EVoC clusterer, no 5D needed |

**Representation pipeline** (BERTopic):

| Model | Role | Configurable |
|-------|------|--------------|
| `POS` | Part-of-speech filtering (keep nouns, adjectives) | `pos_spacy_model` |
| `KeyBERT` | Semantic keyword re-ranking | Requires local embedding model |
| `MMR` | Maximal Marginal Relevance (diversity) | `mmr_diversity` (0–1) |
| LLM | Final topic label generation | Provider, model, prompt |

Default pipeline: **POS → KeyBERT → MMR → LLM** (sequential). Parallel models
stored separately in `topic_aspects_` for comparison.

`MIN_DF` guidance: `max(1, min(5, n_docs // 100))`. Small samples need `min_df=1`;
large corpora benefit from higher values to suppress noise terms.


### 5.5a LLM Prompt for Topic Labels

The prompt strongly affects label quality. A **domain-specific** prompt that names the field
and provides terminology examples consistently outperforms the generic default.

Available prompts in `ads_bib.prompts`:

| Constant | Domain | Use when |
|----------|--------|----------|
| `BERTOPIC_LABELING_GENERIC` | Any scientific corpus | Default — works broadly |
| `BERTOPIC_LABELING_PHYSICS` | Gravitational physics, astrophysics, cosmology | GR/astrophysics corpora |

Add new domain-specific prompts to `src/ads_bib/prompts.py` and import them here.


In [ ]:
from ads_bib.prompts import BERTOPIC_LABELING_PHYSICS as LLM_PROMPT

# For domain-specific labeling, switch to a specialized prompt:
# from ads_bib.prompts import BERTOPIC_LABELING_GENERIC as LLM_PROMPT


In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
TOPIC_BACKEND = "bertopic"  # 'bertopic', 'toponymy', or 'toponymy_evoc'

# BERTopic providers: 'local', 'huggingface_api', 'openrouter'
# Toponymy providers: 'local' (HuggingFaceNamer) or 'openrouter'
LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview" 
# Quality alternative (slower/heavier local labeling): "google/gemma-3-4b-it"
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")
BERTOPIC_LABEL_MAX_TOKENS = 128
TOPONYMY_LOCAL_LABEL_MAX_TOKENS = 256

PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL
TOPONYMY_MAX_WORKERS = 10

# Adaptive min_df: scales with dataset size, capped at 5 for large corpora
MIN_DF = max(1, min(5, n_docs // 100))
# ──────────────────────────────────────────────────────

### 5.5b Fit Topic Model

Run the selected backend (BERTopic or Toponymy) on reduced vectors.
This is the most compute/cost-intensive step of the pipeline.

In [ ]:
execute_stage("topic_fit", run_topic_fit_stage)
if globals().get("topic_info") is not None:
    display(topic_info)


### 5.5c Build Topic DataFrame

Merge topic assignments, 2D coordinates, and embeddings into the main DataFrame.

In [ ]:
execute_stage("topic_dataframe", run_topic_dataframe_stage)
if globals().get("df") is not None:
    display(df.head(10))


### 5.6 Visualize Topics

Render an interactive topic map from 2D coordinates and topic labels.
Use this view to inspect cluster semantics before curation.


In [ ]:
execute_stage("visualize", run_visualize_stage)
if stage_enabled("visualize"):
    logger.info("Topic map artifact: %s", run.paths["plots"] / "topic_map.html")


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [ ]:
CLUSTERS_TO_REMOVE = []

execute_stage("curate", run_curate_stage)
if globals().get("df") is not None:
    from ads_bib.curate import get_cluster_summary

    display(get_cluster_summary(df))


### Checkpoint: Save Curated Dataset

Save the curated topic dataset as the handoff for citation analysis.
This provides a stable artifact for Phase 6 and external reuse.


In [ ]:
if stage_enabled("curate"):
    logger.info("Curated dataset artifact: %s", run.paths["data"] / "publications.parquet")


In [ ]:
if globals().get("df") is not None:
    display(df.head(10))


---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.
These parameters define which citation structures are constructed.

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────
CITE_METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 5,
    "co_citation": 20,
    "bibliographic_coupling": 20,
    "author_co_citation": 10,
}
AUTHORS_FILTER = None
OUTPUT_FORMAT = "gexf"  # 'gexf', 'graphology', 'csv', 'all'
# ──────────────────────────────────────────────────────

ctx = current_context()

### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.
Outputs are exported for Gephi/Sigma/CSV workflows.


In [ ]:
execute_stage("citations", run_citations_stage)


### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format.
This supports downstream tooling such as CiteSpace/VOSviewer.


In [ ]:
suffix = "_filtered" if AUTHORS_FILTER else ""
if stage_enabled("citations"):
    logger.info(
        "WOS export artifact: %s",
        run.paths["data"] / f"download_wos_export{suffix}.txt",
    )


---
# Summary

Final dataset statistics and output file listing.

In [ ]:
logger.info("=" * 60)
logger.info("PIPELINE COMPLETE")
logger.info("=" * 60)
publications_count = len(publications) if globals().get("publications") is not None else 0
references_count = len(refs) if globals().get("refs") is not None else 0
curated_count = len(df) if globals().get("df") is not None else 0
logger.info("Publications:     %s", f"{publications_count:,}")
logger.info("References:       %s", f"{references_count:,}")
if globals().get("df") is not None:
    logger.info("Curated dataset:  %s", f"{curated_count:,}")
else:
    logger.info("Curated dataset:  n/a")
if globals().get("df") is not None and "topic_id" in df.columns:
    logger.info("Topics found:     %s", df["topic_id"].nunique())
logger.info("")
logger.info("Output files:")
for root, dirs, files in os.walk(run.paths["root"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        logger.info("  %s (%.1f MB)", fpath.relative_to(run.paths["root"]), size_mb)

logger.info("")
logger.info(tracker.compact_summary())

In [ ]:
logger.info("Building and saving run summary...")
run.save_summary(
    cost_tracker=tracker,
    publications=globals().get("publications"),
    refs=globals().get("refs"),
    curated=globals().get("df"),
    start_time=_pipeline_start,
)
